# dcgan-wrapper-netG-netD — worked example 1: Wrap an encoder and decoder as .enc / .dec with no forward

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dcgan-wrapper-netG-netD`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

The DCGAN wrapper idiom is just an `nn.Module` that holds two sub-networks as attributes and deliberately defines **no** `forward`. Callers reach in and invoke each subnet directly (`model.netG(z)`, `model.netD(x)`). The same pattern generalizes to any pair of networks — here an encoder/decoder. The only hard rule: `super().__init__()` must run before any submodule assignment, or PyTorch's `__setattr__` raises `AttributeError` because the `_modules` dict does not exist yet.

## Worked solution

**Step 1 — subclass `nn.Module`.** We want PyTorch's machinery (parameter registration, `.parameters()`, `.to()`, `state_dict`) to see both subnets, so the container itself must be an `nn.Module`.

**Step 2 — call `super().__init__()` FIRST.** `nn.Module.__setattr__` intercepts attribute assignment: when you assign an `nn.Module` it stashes it in `self._modules`. That dict is only created inside `nn.Module.__init__`. Assign before calling it and you get `AttributeError: cannot assign module before Module.__init__() call`.

**Step 3 — assign the two subnets.** `self.enc = encoder` and `self.dec = decoder` auto-register them as submodules. We pick `.enc` / `.dec` here but the mechanism is identical to `.netG` / `.netD`.

**Step 4 — define no `forward`.** A container with two independent networks has no single canonical forward; the caller decides which subnet to run. Omitting `forward` is intentional, not an oversight.

**Step 5 — verify.** We instantiate, then confirm both subnets appear in `list(model.children())` and that `model.parameters()` covers both — proving they are registered, not just plain attributes.

In [ ]:
from torch import nn

def make_codec_wrapper(encoder, decoder):
    class Codec(nn.Module):
        def __init__(self, enc, dec):
            super().__init__()      # MUST be first
            self.enc = enc
            self.dec = dec
        # no forward on purpose
    return Codec(encoder, decoder)

encoder = nn.Sequential(nn.Linear(16, 8), nn.ReLU(), nn.Linear(8, 4))
decoder = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 16))
model = make_codec_wrapper(encoder, decoder)

n_children = len(list(model.children()))
n_params = sum(p.numel() for p in model.parameters())
n_expected = sum(p.numel() for p in encoder.parameters()) + sum(p.numel() for p in decoder.parameters())
has_forward = type(model).forward is not nn.Module.forward
print('children:', n_children, '| total params:', n_params, '| matches subnets:', n_params == n_expected, '| custom forward:', has_forward)